# Driver-based Financial Model

* Uses Excel as input/output
* Is fully driver-based (revenue drivers → financials)
* Produces a 3-statement-style output (simplified)
* Can be extended easily for real LRP use

## Concept

Instead of hardcoding numbers, everything flows from drivers:

* Users / Customers
* Growth rate
* Price (ARPU)
* Cost ratios (COGS %, OpEx %)
* CapEx %, depreciation %

In [18]:
for t in range(2, months):
    prev_users = df.iloc[t-1]["users"]
    
    new_users = prev_users * drivers["monthly_growth"]
    churned_users = prev_users * drivers["churn_rate"]
    
    df.iloc[t, df.columns.get_loc("new_users")] = new_users
    df.iloc[t, df.columns.get_loc("churned_users")] = churned_users
    df.iloc[t, df.columns.get_loc("users")] = prev_users + new_users - churned_users


In [16]:
import pandas as pd
import numpy as np

# -----------------------------
# Load drivers from Excel
# -----------------------------
drivers = pd.read_excel("drivers.xlsx", sheet_name="Drivers")
drivers = dict(zip(drivers["Driver"], drivers["Value"]))

# -----------------------------
# Model configuration
# -----------------------------
months = 36
timeline = pd.date_range(start="2025-01-01", periods=months, freq="ME")

df = pd.DataFrame(index=timeline)

# -----------------------------
# Build operating drivers
# -----------------------------
df["users"] = 0.0
df["new_users"] = 0.0
df["churned_users"] = 0.0

df.iloc[0, df.columns.get_loc("users")] = drivers["initial_users"]

for t in range(1, months):
    prev_users = df.iloc[t-1]["users"]
    
    new_users = prev_users * drivers["monthly_growth"]
    churned_users = prev_users * drivers["churn_rate"]
    
    df.iloc[t, df.columns.get_loc("new_users")] = new_users
    df.iloc[t, df.columns.get_loc("churned_users")] = churned_users
    df.iloc[t, df.columns.get_loc("users")] = prev_users + new_users - churned_users


In [21]:
df

,users,new_users,churned_users
2025-01-31,10000.000000,0.000000,0.000000
2025-02-28,10300.000000,500.000000,200.000000
2025-03-31,10609.000000,515.000000,206.000000
2025-04-30,10927.270000,530.450000,212.180000
2025-05-31,11255.088100,546.363500,218.545400
2025-06-30,11592.740743,562.754405,225.101762
2025-07-31,11940.522965,579.637037,231.854815
2025-08-31,12298.738654,597.026148,238.810459
2025-09-30,12667.700814,614.936933,245.974773
2025-10-31,13047.731838,633.385041,253.354016


In [22]:
import pandas as pd
import numpy as np

# -----------------------------
# Load drivers from Excel
# -----------------------------
drivers = pd.read_excel("drivers.xlsx", sheet_name="Drivers")
drivers = dict(zip(drivers["Driver"], drivers["Value"]))

# -----------------------------
# Model configuration
# -----------------------------
months = 36
timeline = pd.date_range(start="2025-01-01", periods=months, freq="ME")

df = pd.DataFrame(index=timeline)

# -----------------------------
# Build operating drivers
# -----------------------------
df["users"] = 0.0
df["new_users"] = 0.0
df["churned_users"] = 0.0

df.iloc[0, df.columns.get_loc("users")] = drivers["initial_users"]

for t in range(1, months):
    prev_users = df.iloc[t-1]["users"]
    
    new_users = prev_users * drivers["monthly_growth"]
    churned_users = prev_users * drivers["churn_rate"]
    
    df.iloc[t, df.columns.get_loc("new_users")] = new_users
    df.iloc[t, df.columns.get_loc("churned_users")] = churned_users
    df.iloc[t, df.columns.get_loc("users")] = prev_users + new_users - churned_users

# -----------------------------
# Revenue model
# -----------------------------
df["revenue"] = df["users"] * drivers["price_per_user"]

# -----------------------------
# Cost structure
# -----------------------------
df["cogs"] = df["revenue"] * drivers["cogs_pct"]
df["gross_profit"] = df["revenue"] - df["cogs"]

df["opex"] = df["revenue"] * drivers["opex_pct"]
df["ebitda"] = df["gross_profit"] - df["opex"]   # EBITDA = Revenue − COGS − OpEx 

# -----------------------------
# CapEx + Depreciation
# -----------------------------
df["capex"] = df["revenue"] * drivers["capex_pct"]

# Straight-line depreciation
dep_years = int(drivers["depreciation_years"])
dep_months = dep_years * 12

df["depreciation"] = 0.0

for t in range(months):
    for k in range(dep_months):
        if t - k >= 0:
            df.iloc[t, df.columns.get_loc("depreciation")] += (
                df.iloc[t-k]["capex"] / dep_months
            )

# -----------------------------
# EBIT / Net Income
# -----------------------------
df["ebit"] = df["ebitda"] - df["depreciation"]

df["tax"] = np.where(df["ebit"] > 0, df["ebit"] * drivers["tax_rate"], 0)
df["net_income"] = df["ebit"] - df["tax"]

# -----------------------------
# Cash Flow
# -----------------------------
df["cash_flow"] = df["net_income"] + df["depreciation"] - df["capex"]

df["cash_balance"] = df["cash_flow"].cumsum()

# -----------------------------
# Save to Excel
# -----------------------------
with pd.ExcelWriter("lrp_model_output.xlsx", engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Model Output")

print("Model complete. Output saved to lrp_model_output.xlsx")

ModuleNotFoundError: No module named 'xlsxwriter'

In [23]:
df

,users,new_users,churned_users,revenue,cogs,gross_profit,opex,ebitda,capex,depreciation,ebit,tax,net_income,cash_flow,cash_balance
2025-01-31,10000.000000,0.000000,0.000000,200000.000000,60000.000000,140000.000000,50000.000000,90000.000000,20000.000000,555.555556,89444.444444,18783.333333,70661.111111,51216.666667,5.121667e+04
2025-02-28,10300.000000,500.000000,200.000000,206000.000000,61800.000000,144200.000000,51500.000000,92700.000000,20600.000000,1127.777778,91572.222222,19230.166667,72342.055556,52869.833333,1.040865e+05
2025-03-31,10609.000000,515.000000,206.000000,212180.000000,63654.000000,148526.000000,53045.000000,95481.000000,21218.000000,1717.166667,93763.833333,19690.405000,74073.428333,54572.595000,1.586591e+05
2025-04-30,10927.270000,530.450000,212.180000,218545.400000,65563.620000,152981.780000,54636.350000,98345.430000,21854.540000,2324.237222,96021.192778,20164.450483,75856.742294,56326.439517,2.149855e+05
2025-05-31,11255.088100,546.363500,218.545400,225101.762000,67530.528600,157571.233400,56275.440500,101295.792900,22510.176200,2949.519894,98346.273006,20652.717331,77693.555674,58132.899369,2.731184e+05
2025-06-30,11592.740743,562.754405,225.101762,231854.814860,69556.444458,162298.370402,57963.703715,104334.666687,23185.481486,3593.561047,100741.105640,21155.632184,79585.473456,59993.553017,3.331120e+05
2025-07-31,11940.522965,579.637037,231.854815,238810.459306,71643.137792,167167.321514,59702.614826,107464.706688,23881.045931,4256.923434,103207.783254,21673.634483,81534.148771,61910.026274,3.950220e+05
2025-08-31,12298.738654,597.026148,238.810459,245974.773085,73792.431925,172182.341159,61493.693271,110688.647888,24597.477308,4940.186692,105748.461196,22207.176851,83541.284345,63883.993729,4.589060e+05
2025-09-30,12667.700814,614.936933,245.974773,253354.016278,76006.204883,177347.811394,63338.504069,114009.307325,25335.401628,5643.947849,108365.359476,22756.725490,85608.633986,65917.180207,5.248232e+05
2025-10-31,13047.731838,633.385041,253.354016,260954.636766,78286.391030,182668.245736,65238.659191,117429.586545,26095.463677,6368.821840,111060.764705,23322.760588,87738.004117,68011.362280,5.928345e+05


In [29]:
with pd.ExcelWriter("lrp_model_output.xlsx", engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Model Output")

print("Model complete. Output saved to lrp_model_output.xlsx")

Model complete. Output saved to lrp_model_output.xlsx
